# Read in data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')

import nltk

# Downloads
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')
nltk.download('vader_lexicon')

In [ ]:
#Read in data
df = pd.read_csv('/content/Reviews.csv')
df = df.head(500)

In [ ]:
df.head()

# Quick EDA

In [ ]:

ax = df['Score'].value_counts().sort_index() \
.plot(kind='bar', title='Count of Review by Stars', figsize=(10,5
), color ='blue')

ax.set_xlabel('Review Stars')


plt.show()

# Basic NLTK

In [ ]:
example = df['Text'][50]
print(example)

In [ ]:

example = df['Text'][50]
tokens = nltk.word_tokenize(example)
tokens[:10]

In [ ]:
tagged = nltk.pos_tag(tokens)
tagged[:10]

In [ ]:
entities = nltk.chunk.ne_chunk(tagged)
entities.pprint()

# VADER Sentiment Scoring

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
from tqdm.notebook import tqdm

sia = SentimentIntensityAnalyzer()

In [ ]:
sia.polarity_scores('This is pretty cool')

In [ ]:
sia.polarity_scores('This is nasty')

In [ ]:
# Run the polarity score on the entire dataset
res = {}
for i, row in tqdm(df.iterrows(), total = len(df)):
  text = row['Text']
  myid = row['Id']
  res[myid] = sia.polarity_scores(text)


In [ ]:
vaders = pd.DataFrame(res).T
vaders = vaders.reset_index().rename(columns={
    'index': 'Id'})
vaders = vaders.merge(df, how = 'left')

In [ ]:
# Sentiment scores and metadata

vaders.head()

# Plot VADER Results

In [ ]:
sns.barplot(data = vaders, x= 'Score', y = 'compound', color='blue')
ax.set_title('Compound Score by Amazon Star Review')
plt.show()


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
sns.barplot(data = vaders, x= 'Score', y = 'pos', ax=axs[0], color='blue')
sns.barplot(data = vaders, x= 'Score', y = 'neu', ax=axs[1],color='blue')
sns.barplot(data = vaders, x= 'Score', y = 'neg', ax=axs[2], color='blue')
axs[0].set_title('Positive')
axs[1].set_title('Neutral')
axs[2].set_title('Negative')
plt.tight_layout()
plt.show()

# Roberta Pretrained Model

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from scipy.special import softmax

In [ ]:
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

In [ ]:
print(example)
sia.polarity_scores(example)


In [ ]:
#Run for Roberta Model

encoded_text = tokenizer(example, return_tensors = 'pt')
output = model(**encoded_text)
scores = output[0][0].detach().numpy()
scores = softmax(scores)
scores_dict = {
    'roberta_neg' : scores[0],
    'roberta_neu' : scores[1],
    'roberta_pos' : scores[2]
}
print(scores_dict)


In [ ]:
def polarity_scores_roberta(example):
  encoded_text = tokenizer(example, return_tensors = 'pt')
  output = model(**encoded_text)
  scores = output[0][0].detach().numpy()
  scores = softmax(scores)
  scores_dict = {
      'roberta_neg' : scores[0],
      'roberta_neu' : scores[1],
      'roberta_pos' : scores[2]
  }
  return scores_dict

In [ ]:
res = {}
for i, row in tqdm(df.iterrows(), total = len(df)):
  try:
    text = row['Text']
    myid = row['Id']
    vader_result = sia.polarity_scores(text)
    vader_result_rename = {}
    for key, value in vader_result.items():
      vader_result_rename[f'vader_{key}'] = value

    roberta_result = polarity_scores_roberta(text)
    both = {**vader_result_rename, **roberta_result}
    res[myid] = both
  except RuntimeError:
    print(f'Broke for id {myid}')

# Combine and Compare

In [ ]:
both

In [ ]:
results_df = pd.DataFrame(res).T
results_df = results_df.reset_index().rename(columns={
    'index': 'Id'})
results_df = results_df.merge(df, how = 'left')

# Compare scores between models

In [ ]:
#Compare Scores between models

sns.pairplot(data = results_df,
             vars = ['vader_neg', 'vader_neu', 'vader_pos',
                     'roberta_neg', 'roberta_neu', 'roberta_pos'],
             hue = 'Score',
             palette = 'tab10')
plt.show()


#Review examples


*   Positive 1-star and negative 5-star reviews


In [ ]:
results_df.query('Score == 1')\
.sort_values('roberta_pos', ascending = False)['Text'].values[0]

In [ ]:
results_df.query('Score == 1')\
.sort_values('vader_pos', ascending = False)['Text'].values[0]

In [ ]:
#negative sentiment 5-star review
results_df.query('Score == 5')\
.sort_values('roberta_neg', ascending = False)['Text'].values[0]

In [ ]:
results_df.query('Score == 1')\
.sort_values('vader_neg', ascending = False)['Text'].values[0]